In [15]:
%env DATA_PATH=../../../data
import sqlite3
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go

from lib.traces import load_trace, db_events, G
from lib.estimate import estimate_roll, quantities

DATA_PATH = Path("../../../data")
DB = sqlite3.connect(DATA_PATH / 'db/srs.db')

env: DATA_PATH=../../../data


In [16]:
roll = 750
df, meta = load_trace(DATA_PATH / f'traces/{roll}.npz', body_sigma=True)
df.head()

,frame_idx,t_ms,dec_idx,n_kp,centre_x,centre_y,centre_z,quat_x,quat_y,quat_z,...,chord_prev,chord_next,arc_loo_resid,n_sup_img_track,sup_run_mask_track,n_sup_img_track_prev,sup_run_mask_track_prev,sig_fwd,sig_left,sig_up
0,0,3804,114,2840,-38.163777,-110.195116,6.377983,-0.129739,-0.680623,0.716559,...,NaN,1.532014,NaN,361,8191,361,8191,0.027954,0.117783,0.092417
1,1,3937,118,2699,-38.250262,-110.379419,6.386351,-0.129471,-0.680325,0.716814,...,1.532014,3.263374,-0.082252,381,8191,381,8191,0.034889,0.136155,0.098873
2,2,4037,121,2737,-38.239896,-110.704498,6.413034,-0.130466,-0.679761,0.717052,...,3.263374,2.443379,0.045918,453,8191,453,8191,0.022031,0.081600,0.077416
3,3,4137,124,2819,-38.353495,-110.879669,6.539965,-0.130329,-0.680646,0.716093,...,2.443379,3.212092,-0.020504,475,8191,475,8191,0.020898,0.066716,0.063214
4,4,4204,126,2838,-38.312638,-111.081858,6.478598,-0.131445,-0.679802,0.716578,...,3.212092,1.943675,0.034392,445,8191,445,8191,0.024538,0.084923,0.094277


In [17]:
px.line(df, x='t_ms', y=['sig_fwd', 'sig_left'])

In [18]:
fs = 10
from lib.signal import unfiorm_sample, lowpass_filter

speed = pd.Series(
  index=df.t_ms[1:],
  data=(np.hypot(df['centre_x'].diff(), df['centre_y'].diff()) / (df['t_ms'].diff() / 1000)).values[1:]
)
speed_uniform = unfiorm_sample(speed, fs)
speed_filt = lowpass_filter(speed_uniform, 2, fs)
px.line(pd.DataFrame(dict(raw=speed_uniform, filt=speed_filt), index=speed_uniform.index))

In [19]:
def fit_roll(roll, n_draws=100):
    """Estimator on one roll, with its events read from the database."""
    return estimate_roll(roll, db_events(DB, roll), n_draws=n_draws)


fit = fit_roll(roll)
print(f"roll {fit['roll']}  s_roll {fit['s_roll']:.3g}  rejected {fit['n_rejected']}  "
      f"bad_loc {fit['bad_loc']}  anchor {fit['event_anchor']}")

roll 750  s_roll 0.794  rejected 0  bad_loc False  anchor roll_videos


In [20]:
quantities(fit['t'], fit['mean'], fit['draws'], fit['events'])

,value,unit,sd
quantity,,,
max_speed,14.658246,m/s,0.092403
max_energy,95.917819,J/kg,0.580830
speed.hill_1,NaN,m/s,NaN
speed.hill_2,NaN,m/s,NaN
speed.freeroll_start,6.294916,m/s,0.065099
speed.chute_start,14.157159,m/s,0.059858
speed.hill_3,7.138185,m/s,0.066968
speed.hill_4,3.769554,m/s,0.070729
speed.hill_5,3.605668,m/s,0.059520


In [21]:
def plot_trace(fit, what='speed', band=1.0):
    """Speed (m/s) or energy (J/kg) against time with a +-band sd interval from the posterior
    draws, events marked.  Energy is the same v^2/2 + g z the QoIs use."""
    X = fit['draws']
    spd = np.linalg.norm(X[..., 3:6], axis=2)
    y = spd if what == 'speed' else 0.5 * spd ** 2 + G * X[..., 2]
    mid, sd = np.nanmean(y, 0), np.nanstd(y, 0)
    m = np.isfinite(mid)
    t, mid, sd = fit['t'][m], mid[m], sd[m]
    fig = go.Figure([
        go.Scatter(x=np.r_[t, t[::-1]], y=np.r_[mid + band * sd, (mid - band * sd)[::-1]],
                   fill='toself', fillcolor='rgba(99,110,250,0.25)', line_width=0,
                   name=f'+-{band:g} sd', hoverinfo='skip'),
        go.Scatter(x=t, y=mid, line_color='rgb(99,110,250)', name=what)])
    for k, tv in fit['events'].items():
        if np.isfinite(tv) and t[0] <= tv <= t[-1]:
            fig.add_vline(x=tv, line_dash='dot', line_color='grey',
                          annotation_text=k, annotation_textangle=-90)
    fig.update_layout(title=f"roll {fit['roll']} {what}", xaxis_title='t (s)',
                      yaxis_title='m/s' if what == 'speed' else 'J/kg')
    return fig


plot_trace(fit, 'speed', 2).show()
plot_trace(fit, 'energy', 2).show()